In [ ]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
y = df['exam_score']
X = df.drop(columns=["exam_score"])
df.head()

In [ ]:
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('categorical', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols)
    ],
    remainder='passthrough'
)

In [ ]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=4,
        random_state=42
    ))
])

print(pipeline)

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
feature_importance = pipeline.named_steps['regressor'].feature_importances_
feature_names = categorical_cols + numerical_cols

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importance
}).sort_values('Importance', ascending=False)

print(importance_df)
print(f"\nTotal: {importance_df['Importance'].sum():.4f}")

In [ ]:
plt.figure(figsize=(12, 8))
bars = plt.barh(importance_df['Feature'], importance_df['Importance'], color='steelblue')

for i, bar in enumerate(bars):
    if i < 3:
        bar.set_color('#ffbdd6')
    elif i >= len(bars) - 3:
        bar.set_color('#003b27')

plt.xlabel('Importância')
plt.ylabel('Atributos')
plt.gca().invert_yaxis()

for i, (feature, importance) in enumerate(zip(importance_df['Feature'], importance_df['Importance'])):
    plt.text(importance + 0.005, i, f'{importance:.4f}',
             va='center')

plt.tight_layout()
plt.grid(axis='x', alpha=0.3)
plt.show()

In [ ]:
y_pred = pipeline.predict(X_test)

In [ ]:
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"\nMSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R score: {r2:.4f}")

print(f"variancia teste {r2*100:.2f}% ")
print(f"rmse {rmse:.2f}")

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.6, edgecolors='k', s=100)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
         'r--', lw=2, label='predição')

plt.xlabel('reais')
plt.ylabel('preditos')
plt.title('predições vs reais')
plt.grid(True, alpha=0.3)

In [ ]:
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].scatter(y_pred, residuals, alpha=0.6, edgecolors='k', s=100)
axes[0].axhline(y=0, color='r', linestyle='--', lw=2)
axes[0].set_xlabel('preditos')
axes[0].set_ylabel('resíduos')
axes[0].set_title('resíduos')
axes[0].grid(True, alpha=0.3)


axes[1].hist(residuals, bins=15, edgecolor='black', alpha=0.7, color='pink')
axes[1].axvline(x=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('resíduos')
axes[1].set_ylabel('frequência')
axes[1].set_title('distribuição')
axes[1].grid(True, alpha=0.3, axis='y')
